# Group B Structured TTS Full Run
This notebook runs the complete HW14 Group B structured TTS workflow in Colab. It is intended for H100 or A100 execution after the dry run succeeds.

What this notebook does:
- Mounts Google Drive and resolves the project root
- Installs required packages
- Synchronizes the current Group B Python sources into the Drive copy
- Executes the full Group B workflow with checkpointing
- Runs the Step 4 verification script
- Packages the Group B outputs, logs, and checkpoint snapshot into a ZIP bundle

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
%pip install -q transformers datasets accelerate genaibook jiwer scipy soundfile numpy pandas matplotlib sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 146.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 148.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 89.2 MB/s eta 0:00:00


In [3]:
from pathlib import Path

import os

import sys


DRIVE_ROOT = Path('/content/drive/MyDrive')

CANDIDATE_ROOTS = [
    DRIVE_ROOT / 'kamp_hw14',
    DRIVE_ROOT / 'Visual_Studio__UC_Spring_26' / 'GEN_AI' / 'HW1' / 'kamp_hw14',
    DRIVE_ROOT / 'GEN_AI' / 'HW1' / 'kamp_hw14',
]


def find_project_root():
    checked_paths = []
    for candidate in CANDIDATE_ROOTS:
        checked_paths.append(candidate)
        if (candidate / 'hw14_scripts' / 'kamp_hw14.py').exists():
            return candidate

    discovered_roots = []
    for match in DRIVE_ROOT.rglob('kamp_hw14.py'):
        if match.parent.name != 'hw14_scripts':
            continue
        candidate = match.parent.parent
        discovered_roots.append(candidate)
        if (candidate / 'hw14_scripts' / 'kamp_hw14.py').exists():
            return candidate

    checked_display = '\n'.join(f' - {path}' for path in checked_paths)
    discovered_display = '\n'.join(f' - {path}' for path in discovered_roots[:10]) or ' - none found'
    raise FileNotFoundError(
        'Could not locate kamp_hw14 on Google Drive.\n'
        'Upload the project to /content/drive/MyDrive/kamp_hw14\n'
        'or keep any Drive copy that contains hw14_scripts/kamp_hw14.py.\n'
        f'Checked:\n{checked_display}\n'
        f'Discovered matches:\n{discovered_display}'
    )


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'hw14_scripts'))

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Module exists: {(PROJECT_ROOT / "hw14_scripts" / "kamp_hw14.py").exists()}')

PROJECT_ROOT = /content/drive/MyDrive/kamp_hw14
Module exists: True


In [4]:
from pathlib import Path
import base64
import gzip

SYNC_PAYLOADS = {
    'hw14_scripts/kamp_hw14.py': (
        'H4sIAJeq0GkC/+19a1MjuZLod/+KOv4y9gnjAZrXOtYb13S7H7ebhgDmzGywjorCLkMttstbZUMzXP77KvVOParKNj3NzJ0TcaZxSUpJqVRmKjOVGmfpNAjD8XKxzOIwDJLpPM0WQTSbpYtokaSzvFYbQ53xcjZcpOkkF1Um2TIcRsPbmJXPo8XtJLkWpWfkJyvIotmI/MO/n9NfrGTxOE9mN6KkN3us8T+H+b3487/zdMZHcPuwsxdGs2jymCd5uFwkaizzZHgXXsf5IozyLIzvo8kyWqSZ1m4ULSLc5u3H/tvPZ6efvl6GZ73Lj62g/9tZ//zTSf/r5UX47tN5Kzg7P/2//beX4fnp6WUrmKTRKCTTHd7N02S2aAWz9CFM8rQV5NF9HEbLUSL+1mvRD4v4GxlatkjG0XAhpkbHFX+bx1kyjWeLMFvOZnEWRHnA/qrVehfn4cnpu/6Xi6AbPNUC8r/6TbS7HYUPt0lOGobzZB5Pkllc7wT1dB7PouRnXrSVT6PJpN5Sra7Dh+h+9z4e7obDxRBakNHE12l697Mo2LqO8njr3w62b/WGQ9ndKMni4aJKZ3FIvsfD28U+LAm0mCbDLM3T8eJnVNKqPddqlwTlromOFZTFIvdAgZKWs819OkxHsa/322Sc3EQzve1NOJ3moiuJHfJti3zbimc3euXb8DrK7qBmvpylP8MPGw93BZVg4v3fLsPj3tfPat638WSShqP0Bhp9hB+tYPoYkA9BkgfD5SJuiw4mk2ga0bFe3sZZHETk/+xbQHogP9PlbCRrX0ezO7Lfwiz+nyXZKtDsUzCL41GwSIMF2ab5mJDfNJ3FjwHdOIR0Sf0ciilFw2ZdpKPoUYKMhkPSxSIcp2SYD8s5wDybxISIgiy+T+KHYHELfw4JfQeE9JP7ZPEYpDOYEG9LWM0omMSLYBoHd2RLBQ+3MWlE9sHskXQbZTcx2RBkyIT0AtbP1nLeZuh713/f++XLZXjR778D0rna220FO7tvWsH+/v6gVrv42Dvvvwt7Fxd9tqlJHWObBz+TpSHdxCNCjnlMlr520vv66X3/QrawoZA202iWjAkaSX25iJSR+BpQHgBrUEe/2sDh6rUvp18/hL1f3n06LQQySckCUlbDeG8df2yTjVyvvf/0G2l48csxaSmg4TnB+JPZKCcMaJx8I1PPl9dk7m3CeOu18/5F7+TsyycyoIowsjiPpvMJEJcOqPbh/PSXM6isb2tClPMw4lzBsRh6hZbe5jrMF9lyCIJqxPeot7lVF0Ea0k0ZAkcohILqUYJjM+r/1n/7y+Wn06/hSe/MNzX2kRYoJg/dXcHvnRBYLbBuApnW2A3zWyIWxmk2ZXOnX99QiZbHszxhu4d/3wth1WllzoPrgxbucbggc6d7LEfD4cyA9R5m6QMU77dwOR0QL/u3A7vwjSjc2dmzS/dEqdbyWRvePUHHOBlSBSPMh1kyp+yISkT2M//ZUYd/fAw5ptvzR76wz2WUsupq7GPJwz4eSPGwEa53bYTtOxAmyg4kqne/Jzavfdg0dsuqmDxkEpD9OGJgvgsOD2XZtl14JIn5e+JwiHAo+UWPSIeLPuGo/fBjv/euD0LlqmagULAo3jn/tYjyu5AoyrGS4hwRU6LaiI+g3RK2G1I2HIeELcffRFkWE7Eez4ZMCxVf52RrJBTj+leAOQln0VRCTmbz5QLVuU8TAmuYzkYJIAJXlHIgixYSBqjAI3dRulxAMybR4AhhFFAx6fhONDc4r+glRGleEEQSXjmUyCO/8wXpV3wghxqgy9pArQxh/rsrr8pDmt2NiSaCUOVCvg+hTmQwLIyWGacwNRG8WqTRg6dkRgRCNEl+j0fu5debGiV204dYSuCh+jOZyWYF6MYYflMFw8vr8MXwrmkkOoZHMRwIwnk6SYaPKyN3SDTb2bqId1SIHkINxfBTQzPtTC9nHzZciL2VSX14u5zdhRNy9lnchrniQosh+UmmY/WJGBOjdhdJz5ZT8vOGCQ8BI11Ek5As8sj4BOcAcnyIM/l9Csjg7Z3w6YGGzgRxL+1znkyTSZQRnSpcpFJsrY3di4+n55fvT89Pwl9Pzz+//3L664XEcqP+obe73QMh6Dm/N1taxWNR0Tiyo0pvRSXreI6q9UU140TeRIPXdP4/ZvQrD+u8d9mnQzra3t4mh7wD+s/uHvlnQE5sv+l13/1CaoN6ftF/e/qVngx3t9vbCi6ctuhSQfEnqKpm+2SSPCjHiOI7wRFXHarU3fPX3dneqHLBKHZWGoZzfnIZwDhj6y+TJF80POpNUzQ9Bp6z/0LMn7LRRXTjV00IJUV3hC4t5UQUxNPreDSi59R0mQ1jrXy4yNKbLJoinYLQb0zPWPpHvfKY/CfOXSXXySy3wLgYFRdfIKZCqilZrbKpOr/Gs5iDqMSW6AIcvNAC2MpMHqvhVlwejwaIEFyKESo8aK1qOH4ZVALfeq9oHcw+BKXS+nMl7HED1OKD1YJZ9YItw6xXx5uGwr3sfaCcSYBu6fbBlm3TG2AY3BLU778jQPZ2ceHFWb/3uX/u5IHqYGTvqQ4dw1zfTvWW3cDaa6TdcLoMo2y4SIahUo9MKIJRlYzh9zhLQyJcFmlWtftROlwCeROlTWsNYMmunajuB3j34JWojv8Dn10S14FlQBJuu31EardByrV3Jf29heqHL7SV51k6JSqQrqp5znXsM9F2yfnSte3tbe7Y0YxzaPvacTS0CxgH0MdYuMkLNjQ1hIcLgoFQKYDaJLVyTeN2sgG6DEdVluExX8TTMJGYYnQYrirUnAzSg7V18YO8ZpshTfuqn2bYFptE18kEFG59UbN4mZPpgMvBQbeeRQDO+lFx1rPz05Mz4K0N1kr5TUBcAYe9WM7SdtCbjVrB8jb4r+Xu9s4e9T18IqrMXRzMk99/j9rB1SRa3tzmg/+a1Rmk4+WCVIkmeRrcEuwGKXVOJISJZGD9D/Ll8BZ8dvNJ9EgdJMkwWETk/2ncrteaeMSfPbKg5REFb0n9889kdn0ww3NvB7S63/05noWC1+3XVTcME+HF5X9+6TsZur71CU8k405mOgOltNfxodCBso0w5GP5xigJXWSOQRZSQ0XQOSFMwsE9PXjQ8A9JKDD16zydkFWbPAaT9F6R0k2Uz0mFEvy0ihD0D10oyTX+1+mnt3332cXk4h0wekn2bbDzTvA1ncXyYOBoyyruO1paJFgO5qAczEH92eS1h5YUHWBWXCCeldbkE9QcxsnJhVCTiIR2lOlCmlQi0hlXovtUKVp4AuDXgyGef+hf6otlSGqCjUN6ynZvUS53yVGOoIEaC5nkkDXkClSEy0i+ACytUAT1WK2on+S84Gcpr7ROFxpl+tHC6zCaqo3icRA+ZAlRKob5PXUINMbJJAbR2qERK61gnMSTEez0vEOPuFeEPQxaAdTlH8DqB19bELEyGDSDrf+gbTt0DgJeex4R7XbRnt6NkqzBfuTdy2wZt4L4G4ET'
        'pnf0Z5M2e0gWt6otBFc06g9kbrP4AQws3Tr5myjLKSi03fpyMd46qjeBZ9wSfjyJO5Jv0ellhMbIDNvvyFB/pR8arJ4+v676s2k0b9N/buNoRFq6Cyny4D+sOIsXy2wmZyCQnREQofCUuxBOsedCK5tRMg5m6UIhhmIubzTVfHnHVwMvGrN10CjAwqhgms2AqFVABYRvS9SeMwSxtk1JYTRQCKy51D0ect9Mg06VzI3TCQRY8CIVUgUt+ceajle9oMGig9onn76+uwjf9S57oBt87Z30W1REdZ3lRFi8//SByJo50b9wlYuzL58um2L0hEf8NzkWkaWbEF2RKOFkTQi2GoDREMR+JyBfgv9Hl4/8AxKETo18RYumGphYrTPVCioQQgU4Cjojp0X2aLUi8Gm1thzZIm3oIVpN1jb+Nozni+BfRJuN+1mWZl5Ics6244VwmrzBPkdZFj12YOUgmEtzKnRAhtOpj8n6LDr6glEtuDEh9KcBaQY/s6oNBCcglLXTbAUHcjxcsydcrKE7N2iHPpTrFUFLgUo25nmt5QxCbbiKK/YuL3winaBum53tN6NnomLW/o+M+WtMo29gK+zuNDWin7F1Yce2AuLnPXp3CtpJrCz+xkZEtiGFbyPGuzqYs0iEyWmKFs2g2w3AuqzQxkdEqLR0goz2Jnnsas33mz4h4bKl6Bw17NEgxkr5EG9yZS3QQGJMhKBkabqguAJZ34D/0G3bCvhpEsodG5kMVRKXkmkEWVozIC7Y3waBQQnfzFplgRYCAgYBGK6PskeIdaxbjVW40BWKpRlARI5o5cBzWfOaqFRVGstNnC5MxJL2eYP9YDiUUkyRGXzidMY8MmmWxLmME+LH5BT0GwUJB0PsoDKYvxlbgarvuqpbAUWoyRtXE0e0EWq052rkCEXSWo2Xk4nVin7kJxsmDgkPpPIgmek4a4NNIkYCn4qA1dZRAyiWczkfwR7mURsyXJdZD9hXkKVs09CPag9pdRQJsK/Xk3R4Z7YcxYsomRD1ETOiVs3YZmoY4F7B0cYNI1q5qUvaJE9m+SKaDeOGHnkM3WmIQ+CfnmtqGkCaqrRNWAzBUbScLBpsH8EB6um5qc2b9LYAZsGK9RYKd6oNoA43Yb/0dlBHtRimwN3ATksxmgt/k4LUviFMt27WIwO9GjQlctRyMDzNLMAaeoySdjQn6uOooWA0a9oa58vpNOLbmjJnc2hGNYbCpgvClepiIKBxmuFdarAZ4TbksI0wKUWWdUqBGqtqGkFRDNIojKAqj2VvmJUsBHcsRBktzHl3zOlq8Vf0L4uurmC6gAk1bY1WrxSFDTA5GZuoEqpywZrMaUwiwkxoiWRelKgdtfhxmlKwo5hOX6IBA9ExIJRTfIEA7WiLCRjCSQ+Sa5CzL9lYFtcJmB2XUodXPeJjcYb6ccOARxjJ0Dn2h25Qk5FsHWpw1lvhoLZOQDm6KnaGt3UCNkO23XAQEDliaJRshsDhlq4AKaO9ESyH27vCh4z2muMUt9U9qkYbzRFA8KUjy2EMqVsNcfgT7tUZgiZ0VBFSz8Mp+u+oyUsfmSuczxiDHdbnrqDC+zr+k6c+dKtdE05PjhoQVc9rNO1+9fDB1XrWWyLIevihiW47kshYbOXkwE0154di0XpDFtCIG/EgR9XFs8knRHB3FR7RCuzTcYcdYzdhHwXKqZ+F4PA/PGsjNNBA8LqcY/2du/oGxFAcwaAdx1J4uRRlRCtzKn9UYyEoPbixkPc6RlWB//pDKQtBFYzqgd4KQ/QTZ1atoVVr6Kjl2N9/CANwbes30R8h+wuPibYN/03vtW5vM0zYJCdLRlbZxHoHOOAYgzeCkXmEb9F2fIktzTp5fYzhBTa1CKgu39gi1rp8c6sw7ArsQkZo/8mYxnUB0/gj+cSxk0+IaOfXyyg2VKhNJkHJ8z5H9w1+IDvgw/jLMAR576ISU5DXMioyBu3eRlX2oC52VGES6Bz1Uixgr1BtsG9xlJ0AjNsXQZcao6qQKjOGqEVl10HKAZhk0GxTV2ejWZVZFZuxMTd6pxdZ8fRocY3rM8YKo6h7vZ1+wcbH9mUgB5tdKAuYSbwOJlC8cYxq5PQMnhQXgPcRKbDNCsZ5xP6on031Gz4Gr+UXd+htT3uC2jWgTgBOVJMc7NraDSFjJyhK5KsAVWElALBBp0h3818xMo6C/qtI5sSMO0kdc5+463ouKhko9dxmem16BxkXmFrsZC0NeeXcHWgjfZcseqTQd+liSaxjWBroQ0ZhOSzDQoWA6aUPV4ZKMTCMvFwE0Ir4Fr5+hoTioaNYMxtBzIGo6liFgWk2Fn4zHpUi0UcLmdOTGd0fyWQ9GXIaGCs8GEmfL+e8HBJbaC1UWLFsClx1+ORX19Bv3cEowfrrcC2NMY5OgIaFyqydzNYJNZDfrcpDT+Whq7K9hnZDsw5SStUaLCexNStcihqyiPKVjGsUmsft82y4+5lH1uXut7MaNbQ2LUQMrSDKqT3W9szq1bBWMvyrhDCg7AlrRzJgKC8X0HD9V0GzlfJjbVTbkF4S3a8zfmTfVd1MgYJaHLhaqPworz4ARGyr17keh67qIpEKqnnkqqlnWXn1KzFMYxBiLDCSdsKDLXloCD9qEnRvt7fNcEtXoKgGCUWENi4f5ywitKVFhzY97XnvTWOY4JJ3DpLF1bABkr/9wwMI+hDXGiMAMUdIVPF0ApkIuUSmorg8blf7XTF+F6qWRfBqQNpJHoorMib5QGsthJcQJA0f1zqzwslF6H6YjEMeD65iytUUSyLL9aGL5vbIK8Wb/8hY8zFhXwuGhkRmhqPrTtrbhpwhaZ7QyLe7+JGcsxZLojCzCu12e+CmBxgNqU5Hg9ubS+khQTIUqsqSNk11m4Dg2dQv6eUyD+JNkjDpEYVyT+NFRDN+0kuFdBAKNb6N4Do0Fk9MwZSkI6ZEp0MQV05KT56ACkj34L7P2QFOyJi1Fl1uxD+bUccEVEON1LaHuQdhhyXrZnjP6CrE07cckE1LAR22DG2yj67eRaN1oV7Il68wsJgKQPjD7oLnZtS4AgbtWtQoyePgnJ32KPNuIBIe1z/AAIIeBRX0Ls4D2R3QzTTJc4KRTvCEu3puB2954F3w6eTsy9YbAoAgP6arDLcGGdy37brsT7taZIxbcatKHGoePcJiEEwCQtrwt+BEmGXSai4/TDGGxri2QAJwHBMN9SYWDrRHrNQRrWflSLhWYERV8a80luYHx8oZWtx6sXLIa2DFw9XrRdFuqHTjWDbbccZuuxtt7Cg340tpxA0atpu9Yf+fI0udMShXXNsq0WNaQ1dY2t/hcC8UDkf/dXq7qM+ZZtts8N3r5A1Me+e5GbRYfisTiaPMzDzCq2zCQPxHcduLvt/z+63emzfc+U1Z8eeGe9aVp8X6Vi1Xi68It8ZJm16IdHHSp5fZ4I6cUQb3cSSVsrGLs0v5IbDkU2bMm0ddQzFJhs7mC6DQcjOtzkVRNibD2z61u3Tl'
        'LfkRsTJ0ox0UahSYYwR4c26sRjjNaxYDwB+8rODDpkKd37rHbl1HEMZ3YjM+KY5zjLnl90tu7JfcD3paM2dwkttf/iIb+0+6KYehyPxcLs8d25EXaAcBduxXiTKYtaArtUnXOUErphe/afYj90iw+2QzrcA2CK/BDKyzB04Sov1aUzk3Er7o64xzi63JX9bhIC/PB+wMZ8alDA64kBHYeb1WZyZG4jQbgJ5V7XswkhfhDN7cYx1rhxWHk9EYKKMFD3qqu/sT8RyuXmRYX0WwL8LguJcfbJrMnskgVeB3esI5jbu5krFpxe58bDqj8zItYXcFk6nXElu2q3wGSpQ+r4NntyE/wAd6jxV7FaXAZWzQXUhlu7rZkri8clYYVDLEIt9a4S5GHbrhDcp3ub9Dx54nXYJHT4+kdGUJ7DjpFVmznPkDOx5C9qsRR0qN0Pace2+VyXm+A0UOSG17OawKyOj8AuYCj/9XZaTsqJG1SncYIIFTKs5gub6ib0HeXFZbICtsTilk7ZmuorYbu84C5snKWUluWsCcWTxNdobSeXYwfX0PKbuBJHU3HRY0dUbjuzmHhT13HlLrIoKHoVjgfKlLi845zjlVUgioQwxbA+kAGoilOBgHcudSAMP8foUwKtsC6fzKXv0ywlK9TnsxDs01RJWqB44RwU2awT+6kqEEDLB21kfmRlrXzk6NvchAA8lsGcuPilOA3lLoUm8FDTMpvPMRG8y2sNdb687K0+UcXyWtyvRFO0WFk81blmGHGiVQjvi0ynwulmdg5g9BipX6YdSqoC/JNbfsLU1ME5sqU6soVKJTd/UVVarqapXo1lXZUqqK2GNRHvdiTljolHj2BWcIBsYMmevxroPVeZeym5ofFMfidwKKuBXtmecnEnEw4q4CC6SRaZd0mzCNHoRUNTxmSOON9LLDCtyvhE8kY8+eAbMJIYytHQrNmfj2ZVkkylnuYZCYjW7IIrGdC2Jf5OSNbPggmlUz3Vos7pgh8WJc7W6DtjlvoMGafRMJdHBcp7FMGD6EM9R3ANJOe5v/s11/xlObZ0kKt3ggmFAW0LBoVz+9urf1jtY6L8OfImiR96ohQNHkty1tSZo8KxWP+tD2gjNQkH4MLSiA7TQjqmRDAWhBnFl3Ek2vR1GQkPNJh/73anvQJP+vrS0Ly40JtktkY/lXLvteldz7ITLvO8g7r6wDLhju7++HPG5WfzihXNiZvjfzPpswI4j80xkZcQ57CK62EKkyhGuLvott0gBXErQqLu+44SgRw7oH8TzMA8TwjVzkV52dQeC6ccFsmf6GUpKxroC16Z128AU1OnuoY+IDMSV6WxW+Oxk3E4S0jyuzbIABOTmcqwcqFxFc+LICNN94KC4Pe0wI6F3qzhXUs17wYgM4tgdgem7QIMzCqgMRdClkxxNKwy6gq2+E5P/5Tzaq5yaCdE30rzuUDo+D5tvteplMRvzNapZHlz04zax2MtO4FebbDaxHsYWxW7LEotbuN7G5kS+GHcni/9GD5ERmtXhS+G/iaVtRcXfP6h+V7/A3bMWuslIbq/hed4KJltiiEIB0naYTwAFc6i4y3+sYENHx6GOzEGmiia9cKQ7oswyj5dTqboxr0bTwdHIlp11OKxwSz4qJR2UdeqwB8GbeeZnnWea6pCHO5E+zmA2Jv0fsypFfdXgFICoMlaeANXNcu/Jxa/VoTgLIxxrPREJqGQSs0XnwH6iBGf6r7rY0xvVz9iZHPAqeNAjPDBxPbJ6zjPmRHG5KFGIYyZPezXO7Lgdj7iwyIn0f+gdUt1oSDRXoDe7okAFpUKA7tiNmN8AlCGWm04Zc9yaS2hCLS/grSGRSvc3m1cii2U3c0CfRZMHxWjcckHqwAd1e124Iaw9z6dEphM8a2SzsJ3P9b0WzyEqn/9pKkON555lFJlueYztqWXtcEH0nDcaJ8cwc0e74h0FNY7P+DAL4DE6R0jLy5s+CeLacxizDvLFumqGSZ6OHbL3enOzG2TmZjVNSn7dk0jCtq1omdvgWU41JE7g5QfRn+YyAXmhq9koDk1gR4hnxkydLxJv05GVkNol1OFa91UQ4uoUuRxMrSl3iLp7dkPW9NZwWAwcMJ9my7ouQ5xyNm8AZMJeYdkGx9gB3CPAbgWYxfRXCBUbfLp3A/yKIvynfUZ7G/LkQxzrSbWe0oi+I4MrP2i0UcbP7JrmGIyyWX4iVgV1Ms48p0qXSRZwGr2w0kk3x711eo+Qx4KaWH4NAhU3mGxsBGnBvPRIIzQLpgYX15W3MxRjF1NbOXsAUVqJ/MSUGFOqI8p10eXNLZrG7HbBZBcNJMs/ZKYuAUYMgushy9NgO6riv85gsildDZveAiPxMxnRbLajeCjYEJM/MZbNXy4esq44bVxbHFflKJKM1sa5f0n+4whwGmDj9i9uVjOeckK6jv3bU0oip6Wzq05gwFHOsNY+BaSW1czOVs0DdxKqm+uHJbMeqGV+wvYMcMXO43E0tXvCoWkP+5To3SRMZOy+5ziP4JRsMTRwo8Fep4uHPVQ8IgFYXQEdOE1xNs/gV5DRxgWe4o0+90Cxj3HQZ5fLo6stt1A3grXJqfDnVzpxC3jD2Jw6EB45DY/x9j4x6phic1AOtlPatdJmoop8Pk/ljO0nF01gP0T1c065pGWg0UUuWA3QjMgJerw1HInsk/mMiN9PC4umtLEelHjVSdDnWlawI6lOdjj4GZfno6ENA1oU+fkpTVKNd7OUGUUxTAFxJIUwoXV1XESqGVVdNsmvqNrE+z6Z2BRleYKBZ+mAT0U8NNahW4IMToKxBk+g6nnTr2mwoUupWR1cCWwNBovoO1CrynRfPcrjrk99G4Buk2y5v+HhRkWUEpui1QVGAXfrfppuZ2dzTaqNNXuDUwzesphWkkJzHlVk2KBVHWltXhYEZCgbzE7FgioeqSmpCkOXPmPYV25EDW8xpw2DmWp+4s+ZKvw8KZZ97jnrDZzPdhkqn0jAfDvpuOZQc7FblUFo1NYsrswIZhJqMeDxHudwLzTm/kM02n1PfHgPSCZ4UMHGDW/Pfay+6rJhOxg1EPWUYAyC+QvNJNLNXaK2EV870gjpvATXZy3JYbs3mmm8+adl7uiVEiKakt2ZnP+kZ6v/Wf/sLPSKd9M4QIt1cRdXwvI1Tdz6JU+fr4X9yqC6DO9UDRuiFx1ZtxZeO6gj/pGapdVivX8Qki/R2DKOEWVoM02iuisxGiIEarbQyPXGkjmuyH2b6w070hS2dQK60yOIc+22hgNrGqG4DrWn2Gru1XglBuCeAx8mQH+OFHx4DcNUZoMy7Yq9AlLh6KMm+EF6/WMTzYJdwsXEWyTChQFBkO4DyLfBdQof0/jjRC+5ycmqeTAgDiuFVdjiGE2YHiMuBH04ieGo3AXUGUESHSUgxnudtw0tMFLZQSzzSQBfLhM91Uw4k/LicWz9pgFmSk+fKbFvkKGHjfIJ/noWpez1Gh9mW/5lKB8/COdd8LzE2TRuw'
        'jObCjhHLD9j02Dlcjd1uwCZ+q1gaoYtTnrynZiGJad6PlgMGbDPxdL54JCtgjfq5jno1hl/S9bkyJVXs3z1xSRPm+aOlf/FlIzIOtpb6qxv5VLiPlVWotG/xbp+WEqbUQVCSixhVfFO14l5pRZy+iF8HKbpAWrPTzMl5ivCPKyQ1GjTUCZ6DZ3FW0pRLeVS0u62ekROqJdlpXW0TXrGEhgPzBMsgH/shX28G+a0f8nBtyLaF28zvLv7n6RqkW5YxOScHYTXWTCWekbWcbTS67lqEbrUwJufEYt+PxXgdLA5QqA+/jaeIcAy5Cg2axFElaF8KF5X7SUMNKlEbeWdNHEHC5wSPKjCxMmFGhFwL2jRNv44p2ik6aaxwi6vMvfC4d9En/LAffuz33vXPW3geWl+FT80a6mxkpGsWj3/iL64kozLRqDPVFHX01fkZF0xQeLDw9hCZHNOjqyOjqamTzZrmHCW8E8lNwoSUWCyJOVMVMcBVHtMufnnFSbA4AZk87PbCi4+n55fvT89Pwl9Pzz+//3L66wWmX4Ciz1bmYkcosCOpkOO5Sy1keosr7DIdNC0AmhMadG8dS1d6W0dTfmG5q3MAlDJcBYG5eRlCV9eTrN3HyEwfuPRsu5siEyu1SNoAyh24bNq6R7mL0W24m4sGwxHb1X+4q6Onfbp180UfLF/A+DmuPyFsPrPHGp+E2XQR3WDH+bMDnItWuD7jjAGQvMp6INJZTVFQy1tuXsu2uvJoc37aCKqtvGfpmyVIKhIbleXGrmSVRgZ+Q3b0fzvblXJDLs0Pkxn+5wIsuaEGWygzihHhlxtvIktwWF7mQulh1DZgi52lgqJ7eqSAZPCVwqLdTasJF2M4tkAR90UcXYEMu7DFiUcMmRjtlHK1tWQRlke6aOZf6SmfhVXpTDNgBzQ/h8EyCiQTCjAoFFJrCaoNhRVV9LPkJplFk/DlJJcvFJgC9QQbvaDUW0PylYg0QpdPcuzPt7+vKOHcLB0dyK2jhPk86hoHiMqS4I1kgNYbiLYseKPLgjc/VhZYww3fRCUC4U0FgVCGjwKRcP3SZwkC8TsdJlz897qI/27Ed239X82s6ADgYqzMiMK11WE6nUdZkrv46pqs8WU44YacbAUOJjgXwwt/IxVeggQFnmye6tzKXLMi7qS/w6pxpb85ksGRrn8cR5JvRErdQ1cmwaRNbQUQafoJXKpVdUlHS9uiq+KdhXtINwXrnEiOjvqp7DEb14HRI5WcEcnKV+YjlgY/UU9V2i21ZyyNVnSHATOvMysqtY/Tjp5wd53t3dFzSAE9KXDsK8yM8Y9SDmc+7lmkPK5mdcXD7eKfdnU1h676065WYIHdczE3xrHof119CgoCvumgJxevdb0pWsTbCJE7ILsv8HvIeuV3Yu1Vv/K9hAkulp32tsmO97zseA/pih7HVHMFFrwnWY791qzNg/d0Hrz3Y3lw4du4Lg68V4EDl6IDs+CVpKHOcNmzV5hlIRcbnFj9z5L6bWneTQthVHjX0jEMil4ZKdpZ1Va7ZMXdq1608uy5CWu8obgY5ejgyXrsE6PafkwVhd6uiDOdQDDW7JQXqyy5YZIbw2F+or/kVUaCJTFTnlXyhE3x6JOCsKm/Y6E2jIWSUUwGSpCbzO06c1hT3fZSh2Lr1lod3NfNXN1hXWQ3OWZC3XdrOflcM1zb8uvCwgY6ugtXmwgcA57NGtbkU7Y8spMF2xFt0JMZA+pedOdAMcfzBaSZWbP+JLFpx985Nu16g9i0a09s2qoRSPtVKx5UCywqflqXdgixjdCiAanhJkTVTW+kVnoMWul+eHHW733un+tH3J2Bdt0ZxhOVAzqQOTKMttdaW2D11CLEMgRYEGAo0ieiQGHpjyaGI2Jknlr7mZqm5ZOROSilX4ZjhN4Ku+x9uLCaWFCttjY2HZE1GKXeKcDdv6qjPigaNVwMNavT1fKM7boaeuU1qRcZpJlarYAytGHb8XJgfhhrcUau0Agtz6HPbkBP1mIMNKOmChGCWWBGXxxfBbIED8uQE/yiVGEdQXu8bkGqx2atMBar5ggRsN7Ysw8VLrQ6bLL03N915iB1PLfRLUtZCf+jQd/d+jHve4swj4SIQbjzfEFX8nI/YGC2JBh2pYbekdYeUqJXFHcOgruPvwcimxlQPJFEbe+JTTuGAhJuimmLpxWrTFYfNiermwpk5a2jnpBI7+MsS0Zxt8Kl0u9KZjerkdmHKmTGk8StRl0nJxdbl5cXGi3BxX3hDg6IFjLaAg+LpClAJtHTbqP7BF7dhodQsvt4VIm2hEYvs3JuHNRIhv59whqtzKwvFOFYDLZqwCNCpMfsr5xLimPIbczSJska1tN1WuTp73GWhvd0XTAkB/x/QmbDmktMttyKhVR0rGSfrA4LP+naja8cqbQHpRb1KlJRsjBHrsyVbN77fpv3mMtBOv0nAf45fMITd4URWEvVpYcZoiibSHOlFWdnOOeS+rtiY8bgHbWJCtZFiiK/AUK0sBL/4n5xnJ/nHcxV4vsEirv2Mw+uGSuuWjZr55Lw92+6BVRrvR85KIuJxHx0fxU+uu/JiY9YKV80zYK//2oYqDdvt8uQv+9mmPtlDNNgV/xEoQ4ynVL2UqIYvRhnOSjjLDchG7bGWmA+T/AfJ1OBvQv/KdmoBy7Pk/XIZMux3YxU0F3IA93EJH2wCkkf2KnSbWo+0Kn54NVQM1kYGLKR+c5FygduUj5YkZTNc6c6ABs0DfiAMy3zaeITcDuLSYth3Ki3wQQ2rzf/bPvhWt8PMKcnMWG2Pfb39317gxR55KOMEEK//sB9dPz/7z66ZvsIJ71/yQ21nistX05pyuh59Ah6NlmFEq9aAVb+drD9FRxs+6FbRbFJOnTT64/ym+0XACrUKp1z88PysaHmxs6mZ2NLUq6Iuqd7BLofy+3IXDQhb8U8Yr6Eazrolrn3XUnXjCpXfB0hnJSV0LgfKoA10DhhGwZheciG9Jld9qTen8Q79vYFvWO2b1He8JCXdO0QCtZWxYxw04Nd04oIqXl9csMNfHJD2ydnR4VoRpxC2nYFxcimxQRe2GHLgR+T6NfyJB5WrXhUHWKlty007yN/4cARN/s2PDs/PTm7DC8u//NLv+qjFKiN3gPomNazF8zZuMJzF8rfZdiGaUoFUfVfp5/e9rErtEoPZjOtn5cePV1U5VEVsgVYi/LMNosAK6deQ70aKrVp4zq1qAB8DPcm308BqU/4KX/2u1krdgTyd6qwI9Dnx7st9rXwfjEdev0tb6m/5aMyiTOKW8fjclvB43Jb7nGBBeoqnQn8CdSdEO4c3P0R/rzb1RwtH6s4WgiiCKwcsnSzlVnN5aKaB8dkdblzTrlSgmhh+10q++7uKtFTzPNpV6Knz0X+O4YdBlG2Ou6dfya014esO9zHtg4N3lWgwbvXT4N3q9Hg5wo0WBnRJbRICZAt3gvS4Z/Iz6dryj84i0kVpx5XjMR7vASRpsYHNRg6ZbU6'
        'bs6lSWHbw0hUyxePEzIlDIKRYzGE69DUQgwg7HGlYiD0IhuhSla5ji2cplhkz3UZKlvHGyuk614dt/lQXva5hXEQpDyZ8K9+0p7B+mmA7OyVcphUEPCIMVsDqBfdgtb4rLH2RRk9PFdyCjwFPqbLHrOolO9DkivGjyJ2ccZwIwef27r4Z6XcKqthVr2O2BW3eOw3Qyunh5EXgSo89etbW8QWShK20D+ewyeMpmcFo2qKlsPKKVroVj7cMEWLYfw+7NXL8rn49TrxP20DO2gAvXJXkLHAeCO9Kz+sl9zFd2YtxPSTd3jma3udoBh3dfzMaBH6EIY6wdoYtJ7y61ix5vr/4DU+P7E8O9CLpM813fP6cw9X2uESZ4gzBgaywxRtfimjHZHLhMxdyEb2ZEC/+sn4sIGcKVL8kZxRSHJjGOn95oDremlV2SSVir9Fkymayhbjb2H0GoTRcUVh9LmgnnnQ8+wsLejtLyWTjqvLpM/VZVKRHHGInU1wvoFYQu8wa7c93e80uxaCvQ1gmCfla8eoPZZFrEhJoyf+AO5PDOU/DdqT9CHOGk2aCImcymQF+UKulEmyKflBPtYdHTEU5Q4BpZ/2PMEi4m1e7V3z4mgRfNj0BYpoKPCZX7iwYwPQ3yYWdmFP8IjegD3gbFccpTw9Sxced2noTWQRvN8ODhVHc00A6E1tIeBoqz84yN/Q0d9z1+E53yZstgJffc8T77t729vbrmk4nkvv4pfVdejJTApJ79vqNheDTaGTodoanCw7r0/+V6N5gxD0SW2kD2BAf5RWoG1IohugMfyEdTlgPa9NedDHu4IK8bZchZDvp1uPSxQdbPV9owtHmpikujpi8z4lFUtAvYQ2gg2IK1iYD6WF+ZAKGmRc5g5ILcbt8FWYlNlQSyLaDt2W5MMyS7IQt0dkOZZ5rOfstx3iT89Ip2DqBo8ktd3qZood9p1LBKzzNYN/dKnaZzNdeGU0mS3jImgmCTJwTBzvrwBTl19otExaB1s7FDJy71LXCzit1xu6xsopbLcdoiJsYzERCD2ghN01CKfxIgrhzUK+FRsOCwKM1KHPMABlpxkYZJZOCPNOJpDUiJ24+bpQNXFvt+5K4EQGP2Ij09miby9ghnBU6hg58jpGUCBAC/n0ZTSCQddy/ZARSQYwm2uvwgwgXEFGGXR8KIY1klFLOMaOoqlhRzPovWlAvNmsNr0lhdRyhY+CLLN+ry9bGoWkn4PimzrOu1RHWoB3kSCtfM24WB45VmvljVbZ2GBvvPcFU3RvwLVmGk8s6hVxMZWJlwV1fm+6rXDd4DWR7I1Jsuz8jITMyckFlTGF5Kwu3hmtSraAVPLtxnougj/Jdviw8nYAxIX7+/shvzntukBQeU/oAWJl+8IQ11Qh+H7bYgXL/0pbY73Inz9oc3H8CpWjcJNJTW61XSabFSUgr+hzeJU76vPKO6pUw/Ocvq3Ud+7Z06TR2idzU7qgVAw+73+bB0ec3jrBkyQ8GYn+AragFe1Ahg2oaEtqVh8NPTxRtdfaY1t6UOOqhh5s5DkqM/JIA4/CsSUIK1tzrJt1R66bdcNQlOgzbKm1bLmU/5axYE3zuLGC/eFI2h+OmGHAumjHmYpmhDh6FUYIbbwllogjtyVC4QmKzIxvxkL/gMt3Hkz9ffHur3Dx7jB0W8scZO4m5h918e6wAJDTmOmckx+GnyGVZ5RUV4A48MIbQs3vnS8S3z1a99Lf8E966W8aJfyNeEeCfe2xeP3yX9GtPgrKPmRY6YE1XseSAnecT0Z3qUe6VekueKcsx2cBMMTFO0VXIU0gz3jmcJZD0zJR4J+lD5A9UT/Mgnn7wOOp+0F7MFGroCVz+nqi/1KVuAZWcypJw5COJQyBEsOQjyCZ0ssEUXYzj7KcP/pL/wRKFJ/bvexmCTbeM1rSGMXs/EGv2Zyxl87hTedkRjPcBR9/3dkLlGWYo4LJZIjMo++ct/kFLdZbOxoR4cO7adS3tmgbonYNb+H8mnevcGpr/518J74HrYBbD2gepsKeucNNdWzemdWAqb1KIRIwEC7BAdN/ALR4IXaegR4ILKc9Wk7nZHklV+hCtTb9k3ED9oFtggAegJkturug29bCkOhasJz8/ly9d3Eenpy+63+54IpRHe5hGF+E6Vt8EDn16H0+8fF6mUxGIVda6JNF+fIaTCW8nOpwof5wRa6Vum79ijJjO9qfPckV6u6NIUoBgeTvwf8CP/rbU20aAQA='
    ),
    'hw14_scripts/hw14_experiment_runner.py': (
        'H4sIAJeq0GkC/+09a3Pbtpbf9St4uR9KdmnFTtPcXc+qs26S9mamaTON251Zj4ZDkZDEmiJVkvKjXv/3PQcvAiBA0ZaT3u4mH2KRAA6Ag/PGAbisq40Xx8tdu6tJHHv5ZlvVrZeUZdUmbV6VzWSyxDrLXZm2VVU0okpR7+I0SdeElW+Sdi2KVmnGXm7hZZEvxPv38MgK2tttXq7E+7PyNvLeJVt8F3kfyO87UqZkwkt/a6pyIh7K3WZ76yWNV275wNbXJy/ipEyK2yZv4l2bd0NMq81215K4JTdt3OSbvEjqvIW+RME1qeOU1JFXVvUmKfI/eN1lVccb0tZ52iidZEmbGB0UJCl323hTZaSIvBVp42SX5VXcAPIaBHsd500VeU1yRVgR/91sSdrW1apONvEyXwHyeQHtP6nbfJmkbeS1+YbUk8m7tz++/hC/Pjs/+/DmPP7x7N0bb+b576vi9uzts01eZs3JC9+o9eqnH797+z3WI+XR2S+i+MP7H96e49u2TvLSn7x+893ZLz+cxz+/+XD27v0Pb17HP5+dI/iTl/Hx8fFkcvbh5/jdT6/f/BC/ff0B3t9NPPjnr5Lnx0l8vc5hKnW8zbekyEvin3p+tSVlkj/jRUcNYLbwo67VIr5Orp5fkfR5nLYptoC5kkVVXT4TBUeLpCFH//7yeK02TGV3WV4D/sZ0lsk2RVWuYGE3Y1rRBSLpuv06TpoaW2zytK6aatk+00qiyf1kcn7+wYWhZQeobRsHICyJrG2uqhRIyzWAdQ6kk5Rq21W82TSiK4lWeHcE745IuVIrr+NFUl9izWZXVs/woY+Ky4FKMPfJf0pBEGySmwZ4aHYSTjKy9GLkB0YmgjyC0Dv6Bvn9lHbAZEGdlA0uDKklY4n6E1qtJiCdSvkyoC/pAJNdW4HkydMjhpQjIItqVeYoufgk8B/lz5lGyBcuAp4rzZKbuCTXcVtdkrKZPT8+ZmXhuGkvYpQzVUnKtmETb3fbglxQcQf/zfcg4b+AGX4FZviuql+dv4rk8/u6SknTVDXDzlY8AuX1qkwRerwFDCK3kyywIMHgx3k4kThTQLJRHAhvSq6SIgi1VRVDjViVcbhND8UtW3acVFVmlF6S4ntSkppqvUhUGMK1UWMManoSzET28LAO7uLJ8J/907K0ReDPH8S25EDS+kCHff41LCL/WZ2DUo9kwQBN9aqMWXFDWRkk5RjOgYCfjJCWe7D9UJTj3M4r9qJD+T9AU36flE+0Bpq+v7DpefcaqON7PGBuFyig+QwfCFLYF66l5eUjVjsSdcct++pAJvs1b5t3rF/8eY4aGvrhy9mKR0CQVjwGPZ0JZayi7PPxUCyIlGMdZJvnCv7QBFPRR5vFeXbqNW39CFyegcTluMSfA6yhFfewIMahY01CH1N/nFhBTKDrVTPHi6GKk1GWp+0F4EGducUQndI2MYXC+7PabUP10pH1slG9kpHQliPrrUbUMyjJUo17uLCOmq+rl174tAnJ4gTMDajL/V59IXllvn6gUWIMDwT4X1zkl4SSr/c/NEoAf36EMdEVVZ7ZcuZLTzby8kYpUXrDl4x+sfmMQun6CmXRdHMJdhKU1IiB2Xm9A0+c3ORNG1eX9FGnRgxh8Bk0xW6VL28DINkd6ZgP/rLRpOsEmGzmXcwZ/wEL4SsvLz3aZFpU1wQQ3Y0dZoY1pnmTFOVuoxZJgNNkC85rFuBDKIuBvJa8w6TM2K+Lo5O59zfw9GN/AA6UajP0/elvVV5S+E04hdnkW1rJg/H7IjLhCyS0FWdCGt0IWPgjqevk9hQ5kGKk3E7LjL2jPdGfSCbbadLQB7VdaNZpft8R8gcJlFKYLH0CuPnGm828426GFugXrHLeEsDp3ACRtbdbMr3MAW2wMnd+7keev/PvO4hNmhQEIIIwDpJFEwDkPC+XVaAACKebvAwjz1GW3IS9AbIaSYNVEOayqJL2q+eh98yjPwPWL6D9hDUmRUNOR4NRF5XWkXSLcSaMqsVoWtdZUO3a7a5Ff+FU5bbIK5IFKShlRx6reyridJ2MFURv8mgHdT+TCrKKObcqbQEZ/p3kNTqi8H6Kw/dp037ULNCgRVQlBGz4sEAgd7Bxn7VhCnrLUCAMRSUj8ibZgGaNwRsjTH2dMht1mRTFIkkvT4GEWhi/PahGMQUVTqU8WFcFmExxmWwIEl/gr6SvB3K5XOYrJEb+S5EGrB10BENLWhg4t8EUeBGToKps4a1gNcqqpcVUWAgYrBj6U2bpczC6AOEYg6nwRlOlyeO7BLJ6eKeikbaWWEmsiVxGoJGari3XxpRqOPF3wkqKbxh93uRl0yZlSng1GaYOe7SM5MMqTWF2gY/AcfH8MBwAJ4LdIUULe0t/Iros9QPKdovbljTsDxOKQ8O5OJ736JyVSMzUhK+fXYhHIIPyVV4mBcU0JfLIa5MaGUO+GRT1A3qiQw8smtZRiJId3ypd9WfKZJu0cJs0395Om3wFUISFK+e3rYpbZtZm+VXOjNpVmgX9rqNex2yc1BYyi7xnzwRABry6Lnk1Y0JGRT4FbXxMc0TQU0QBhTbZzhcOHjOMtnF8otRSzSjFGegWhs5M9QkYxvJKYAvgLfOCB2o0HovkevI605okmdK5Rmi9NaerzRBrsK7V53kRKpOkmxx0C6YBrNOKmQ6FyV7FcNRjUryp4vXQ3RgKnJdxymAPMEm1LOhvwYCyBzk7s+26RF4DA2tnypZLaADnv6ZJloGoL3abMvDF9GCu5AaERwFmaAAe24oEBSkDXhyGnUjRMKCK2Y5N+l2mQFCyT7o2fsTwoWN01l+pUFthiTdloWC4Gfh5bL3IDaVr90IN68qerXFqIHGANih0G4ZoAZpSFkK0zW7arHfLJUwiBEmqTdUyR8LWjovIv/K0L2Q1Pqdw3tlD1EghNMIaK6JcCMUGjC0Ym7CGXh5LbaHYCSNsJVObWBm5z8FtfasY7y3IqI3JzZpl4Rf5os5ZUIzv6qnFjJXZfumUerHTr14em5VoN4BR5kJqZW29A4aryaZq0cvPiFGHW/c3Kdm23jmI+jd1XdX7ptAftXOkxuhCjia2JCBZVtToVrQaX8DQ+1J7axp56XpXXsIyo5xSNMy8c3rTXY2OddfJsbR9GTQ0e9nouvlSsBajgbW44CJrDj+wNxFg6wbUucjwpBTqY/lXwCcpzUroRuv1vpnpmNKt0gUM/XIixDFKX46UzkxJcmC6n3cl7uXTlQ38V9WuYDbeYpfDL2Qkj07Lo3zMKP0HXGAW4O1WkOOtmfqhtpUC6Ic1S2FtSvRO2CjCi1Nt7HO7LbEnYJtudsDfaZun8c0VSVvMjkjKfElAOykmxnvqbbHfVHRNp9O5amisdyuwhVa4LR2vdzItZL3EpxhtHSRwhktW9Ee+7UwReIjVaE7nDekAAsNt2FZxns38d+BxrPPfmmcwnSM2nSM+ncZgZuyT6na/2V6C58jqTWEARkUKHBE6E7rbN9ma/X+d'
        'w7j5bKb/nW+/g7+BmFCICS0wonV+pehtIME6JxhAoigNwGZsQcBTXxH5RziNvOEUn6nBEFJKhKcpMEGDPQf+tNze+ordgKTHOxim1B8r79W7X7wzii+P44siqPGuSY1D2bHISQtkK5aAZHJYenRJTDkSvXOdwkwHsgSI4O7gBkVyCY4s2SxIlgHNBPIXDXQzzfH3Fy+OrXpCUwCU9Drr9lv0nd7+1DG8hdJUaouUhdjPDR1YVJlcqKpj1yQNq/Qf3jEGedjDN0wq8T5NF5gu0FusyJZn6auwvTsK4x7VPXh4XrX0qPFIyeVOBXtvWdTG78b2CHKVzcTCY46N6PCCDmxOW67BQCyMhrocE2E7+En5mS9ZwFoynwNoGdwjynpmzIsr0jf0T16VXVcaXTzAoHC2H/AXevWopfEAIWSYH1dJkWeJsQv+ECtDF0tSyjitjj3z2zMVy7DDyfByq4anxjKg6Tlgf+5a9OGFdy4eJYAVJoNhohRoUJBoIosRLR5TDjWTEXRrbxn05hUOTcY2IWQfeGfhnb7kttPfL2WyAKOrrehiUpmtSAI+aE+OEib3+y6vQZyjCBF7zTCGLamhJ9wz8vsExrAKAxUOA2bziWCzGlNmxh2qKv5iyGGSfr1ti48Bl7l3zKrHnvxTFsXu+MDveoXC7kGpgVhsWhgLVJB7Waz8/gGeNxvUhZ+XGMzWQ5xzrhvcMUzWmqOw3pVxswayxL1btPYxPMwShNhKX1f1JdDQtYJNFoAzY3n9qA5zVpmLSdKCbuTtW4iIj1QobIx2niq7AFq1nmdsqwRdw1JgBCxPbxmomUc3FoXAU7cpeluFGqyOxsxSJwFx4HVVUQdb7E52fYYdOsfEM+V6XBIaL1OXR2z86VsfdNAoZelfNLL8OxXKfXynIlLSoHxWYuzUz/+CFX9x73cb7LGFFjkl2tfeE56aEqzjHKBNkW6d0T12DHo7ciudJdR/VXbc5Hug+5JudFiTSXV7heZKBzo+qcUBBTB6XWrKHnjMfWb2yZa062ILqAG6AfwgreOI7PsJBuBQz1hjWz0zb2Qu6kTu7dqRvRAoNXIuFVRybQbKM11PlNnoWRUqhvW8VcWihemgFSzbBjKjXQ0aOuks4pIN8FRC62bmb1v/kDVkLXBi07wUkgi7NzfOOwbH8bMMhi+/ZBPSNW63ynlGXS8KHVxo3PvlIIB9Vzmm92f5ZnZ0ogNgYixT0TRdJG26jllJoPUwQF64TcOByR2cMWRkZt/uI6FUkFAvcfQgIkr/HCJqLsFJwcMVeVLwvG0MfKu6RUhfmhpRJ9f+p6JAETYVtMUynsRbB0HupScNamRDwMzy7iMQXj/xeB/pSYVgJJoeRHjkYYQ3+yvJMFM6OSgoMk8u/P34ECFlJ6ouf+JJqcjMObYkvjBn51dMoRJRkF/KZrfd0gCZR83kI7STEb4kvVPvTjPC7rv4KXcdNDdFXz7VVYk8q82Oraa7bYYLIYd6p+8zaAMA30J71j11X7XtoKr6aNTUpBtU1Z6Nuvo6xSj+To3FG24hT8pl1DdyHZsL9GahAVR3HPgw9JfDLcYOQ29GI6P6q77Z3LkRsvOOM5G3cXcEemU5WoynL3xSJNsGjWe2c+LPlSnfK4FgGIDbEP+bzSfs+5NOANSrdJaK/t0IMLsS9GyclzSwGhkEFHLGMpO6nIln6HRFnslxPGOrS9ZToPVxwkFRAsAaFBVam0HfmhngKACoZ7HPYx3lizp9Tb5jbGzk2vaRw/1e3sCuuxBNza5Akbw3htCLI8ycjlyXhdg5vjPeb3+Drquuq9l+A4OcI2WdVZLrWpJyVeTNOqaJ7Wmd0zhZr0eOPZrQwJuyLC0z40JDPsX+QPWwvxp2Ma1J5JkW1NADG7PuZ1dM+UEshSBRP1KECltiwa13PkMEntn8/uz58RmaWaIdtS2gBNX3vRFvQiAGTyw+84SVJwwv6zM//Dn8sHgMP3x7AD+kn/nByg895+8zRxzCEWgQP4Yf0sfww6sD+IF85gcrPxhu7Gdu+HP0A3kMP7x5JD8gdfBMblyRgzboZNq4a6dOTcmzFD/lztxH3XcTqe9ZLLfW3KcRIj2lXveTo16qopHfOnpLbt9OmbWP4V0x10aNiL07g/AiQro/VPov3pubrffVGQ6RlOzqiOLWu0po8hLu9EPPhMbF6FygI5CnMgBIr9zJ2NwbWMK8KBTQPA/AS+QWsXfy0rv8xx8U1tQ7h1FjGKAh9RXvbZM3G4wpepIRPDzOQpLMq5YK6AaT7VoYacc7Hp7bwSQFCgdH9AWFDjSNPdMujYiifbXsucV62O8RIt4Rt1Nlu0HXLonuJCdNGFr2ZQdC1C4d4IqvaRL/4wlk6742jcTGdzYU3K//MIT2mBCrXTj4p4OSwww12truWSUls8QWmGMwnOWhLfgrCoUK8gVHkcwX56XcEEUw0Tpupnx9Ac3sfbeIuxQf7PmrM98ezhyKp6V8N4FuyIAUagRDPSwn5a+jypLr+FM5UqpSdIiWTy0UnD6LHlBnwgBkgJQHX9B5f3EfdxCU6wX+P6N0QM6ORqoKg6NVsuNAvhw97e/gYmVEvS2tfRgft6/lu2WoS27at7Y6nrywVZi72zO8nWpE+FAY2i7VICClpgpvaNdLhce8LXdtFU+Ikmt6p2CHHNYe35o1U2vN1KjJxsKg9sd1ba2dOmqbsC3a6Fv/Icma+/ehbOl+kcInF5w35urLfTtTavOH7U51LRWNKm4zk6xlCD+7UqXHgPjhn7hRtCrb7sejPsrLR6nHmbhoU963JnpourQ4p9puk+ZSwhFBiYVw10WyhVhimMGiqjDfg+W2H5rOOZBnmJl5hkpuhzulw5lNqFlYQ9mjui0mj1peXoMd18zufEQYUDr+uY+Ma13Utaauhf7KgN2RAK3bPYaRJVNfWYQZrkHQex2qR5/4ORwRZNqTNKnmSrKTa/ISEC4U6Etwgi/m4Z6bGZhty88fZjt2m0bvHh48YQjeuXlYT84GbzzCMn6Zg5Qx+vmdbhSyhnmnAx67kYX0RoPn9D3lBHrLQ9n1qqf8yMGLA40y5YDVvjjBy1c88+3xPBxMbeH762PvTnQd0H2AU6aTIVfkg7Tpd9TIqzvJ0+8Ros+kxCCFuk0Gg2ifJh3GANrz98gKlRvI9B1VcfJgam/IG1SXorogEC0jpdxOsVIgqYfFZTtOcGW5KGPkq0LvgOLD2TODp02R0XTH8OGK7l5vlKoy3IsK1nH/d6BBj8wF7xLcVSViHQO1IfQri2jlfXcWUUlz045vCpV9xY/o3XgkByJ2WWz3HDnaioH0ENHB+C4BkgldOTeyc4p0TPyTb8JxeTrYRgxjf1wB78uo0a0AWpK7Tw+0X0xrabST3115sOcUCLUMu2fLJpT7GgXLJtQ+88/0ZjuY0cRhIHzdlSimwL89Zp/FtiQDOy4uPaHvw7z2ezqhvydjVAEUE3r3+SsxHilLjjCEfZVgsNm7rlGj1t4O'
        'Q94Lkibww6N9Tre3XrXJWxbDFlYKz0Oe7ouCGTtC6tWt5oJJq/gw09tyiXzv6KGVwtWaOsjOjZQQBGRCMmtszcku/fztsZa5635Z1UbXLw+WchuHaRfYLNF6k5Q7DAJDNXooE3+ESnMTf/376/pVZvuPsIcazlkCuTxiwh4D5RRpD4A8LspaiBOj010pbkjkad6WXHcamGJC3ZrAPt7DGZ2wrnziopeqzjkjYCMVJyRzNAoiA0OhCRRvw5gJagiUXsKJGgjmO0COy/t48+7mvtl+K9dhR4+4tDnyelenjDOcGV7oVXKnnsUaFaiSsgUjaipT9+xMaqjin6e133ho78rcpuoifHrIVEjP0aaW+sEUyw2fGhnsaSO4pVes+Xa9UnnX6ImTztVeyM0W3DwQaJum3xOYVjd5Mzu2j3WsWah+RWa6BSndMSnf6R1vYUILw8Lsf6LGNg9z3PjJmrYgbLcvNEwT+lkctPfk53ACxihT+l2fINQYJdKnEan7iKEjU11bmbveARphfrL5MAOU6r3AnEcYORt3193xttowbQ01VMJ/RPjAfSJr1smW4M29++As8nIflGMTyP2kfwmEjj9lnboqexL3Lfn6LvOd34n+kWwhceP6HmNFauFus5EKOvuu4hUjxri6InWdZ+QA+2fwFvaGXrtFsgdZSebl8apxtOofhBu0jcQAeiZRt2uuIk3qdO2tZkMN345gaQ0w+c3H6lvDqJFz/jOMGvMUscWAsZohbOfKpCXcZBu6XXi/efI4+0T5OMHHM02cpoa2uNLKsJBD6PL/7BDsdHqAkfKUga3Rhs6BqtupaMXhdQEftMPBWvfCohJFPEdXih9X3dhFiZ0eBvUT/SACeE2bbfuRdJTSwxg9lVVcKhgbX7puwvnLHsqkza8w+vJnKaXHaRnXmW7zaycWcca/Yzcf9n4v8P/5J/V/8WZO+qmHwcPaco1n8hcfEFvKeJRHayqM5y/w842hpJTMAMM+78hy5VU6Qg+nk7TrpM7oIe745OWlzwKX/VE9SgmJVYv6w/s0ekhiW+wXdeg3o4rKjBWl0UfESF1jTNfSSkv801/8H9VPGrsIzSR99dBCJX9FTbVH8RB6k/Og4rmqclhrVlXNqngStUS6G0w/q6VD1dKlrpaW2tL1Y8qP0lv92zEcYCKt95n68Fkl/rOoxMtPoxLVxYcaGi18Vp6fledfUnmSG1QNdIjt03tsKnR/WBHr8cODtWhzKhPK8GbcuTubwfz6y2PyHuwi1jkD5XNVgoEcwVSq9el3AnTV/2l0v6DSU+WqcHWucpJcoWosL6OSltliBO94etyJAEfKR18EjLlE1BghjyNaVCCMYmj/kt7B1hHJ8MDwHCU7aCfVq+UzQw8foAbVarkMGWiDRtqTGmqjjLWDDLb+1J/ecBttvD34EronMOIMPtijvRyU9gijb4i53fp+IiUn+9wXtqEfWuhkdKgZjU0/2euC2ZaROoe5YtKJj6gYwlHPWmZ7KaU6FDMf0rQ1xZGg+/hOnqkctjyRt5RD0w/fYt1jfT7MAh22Qh9miY6wRvsG5YApKjHlaKEf3jTcBn5+U37mjvse/Fwmv2yQe8wWFFvMVoNwLc24GVbt6lTb+1akuudUFK60Ydue8hhT+FHmcHwnkQakrBvHgwayqtzEqtktYAvHmB8ueKRNPNIuHmEbW7latZQVwSLODjgN6UaxpGv8mAt+8Zcer6Qf42aKGmVzgp86MW6vsBzO/Rgm6bjr6B9r4XezxnkOhag47+wbjsMYNOI9Jqf1+aN306n/BjQ1qdWGVa11s9k1rbeg9zpc5Rn/cAOd4BHOkF6JylcT7NipH9qyhx9qAvYvyVBbD3+Nrn9FJbVWrS0G7oD1ew3kpyuu16Q0l0Kihx1QLW7l57RsS+z+VoPtdP3wvRM4N1sXf5vZ73Sxrc++W1JsHUSe+9oS2xraak8OvjlDFySPPog9dBB45Gns8XdWPPwSCl2g0PmignbM/ePpZh1qL5Rmo5JRtzzEMVjXcYzW6USe6K25MU6/IyeSsPz+5apmycJZkjpLBs4b+P2rydQSB72qVVz3OPWH5zoZ7w+f2NDGYz0ioNbo5cyphdaEBUsFfWOpV8ES3vLt9oAo3ZftDvXm/wtUyYNEXZIAAA=='
    ),
    'hw14_scripts/verification_scripts/verify_group_b.py': (
        'H4sIAJeq0GkC/6VW72/bNhD97r+C4CdpaLTYq4XCgAskqTdkKxoj9ooBWUDQ0tkmKpEaSdtxg/zvO1I/LMVOsKX+You89+7uvaPFpVY5KbhdZ2JBRF4obckUH3vV78Rse73e9Pbm98nVnN3e3MzJ2AcEjC1FBoyFkQajsi0EYVRwDdKau8F9b/LXtAlvo38mdL3rv2fwUIAWuQunbnGl1aZgC2as3iR2oyFlFrd6Xy5fpTGJFkVFIZWFhVLfENXrpbAkORcyCMnZRyKkHfUIflK9Z3ojWR2LvHWGF4tgFSgSxV4uqOdZbrLs/5E4RJtBLI+KieBBGGuwZC7Tbopmq2zDfQqNXQV0PpnNyTmZXsxmI/Kby00uSSMF8biIhh4GmYGXCGZ/XE9PEaCl7gmjwaBbKCWxa2GI2csEUoL1W7SRJKrYuzSefcENZEICw/FBcZpZQHXQ9z6r90vfuksRYkqFdsKuO1SRKkAGEnZuYUzpOwIyUamQqzHd2OXZB4rKGbJG9bJWnw2FVjuD5WSoSODoPonE3gJPQQclJixl4sYAzn6GyTrYkIzHZPCOLOkEpzex2P6gYScuAvfUBr17PMY+lU21Re8/c62hupp9JShxovIiAwuNrqjU8LSmQ2YKgGRth/7YnFx9pmxN9kZVPfwNijY4r2bcUTMm+JMMj6U8gCoZW4Q+efC6HmG0ytQioD9FO76lYVjm/jGmQq4808eaqW3toLL20I8/01xbseSJLY/VKX/j0/7GLM9N29pm4djV+Mdcjd/oanxwtd89JP2B1yE+6Wvc8VVzCy63wZcPpMEj7kUrQFXRBP4Nq2cuAruhWP9SaUfp/pIOdeP/6gG0WbReNNTXRuNL+tSpv8o5Jnf0PPrguPvRefk1oPeukz8l1L3UdZw5FDFgR+TRE/zHwTwYeXIm+4OjUfqlNUrxYZS6hSRKpsIKJV+YLB8NmhVcaKfvY4BEd9TCg2WWr1ybfqGOa/jo/XOdy4P4VL4GcafmcNt3NMt4zo1Tbw1ZpliqVu5hwWXpHvyzAWPp/WHkKsWCmqeEomkM8gWkbmTROSTv9PAa/Dtoxbbol9IvIdsCv68EnvlzPh/W4W1RE7XFBZ5lvl3T6Frx/C1n88kUmb5Obq9/vb66mF/ffBmRi8+fiUsx8xkmn6rXsAa8FkhyjtcUnFbGJM/xIuWHkzF3aWGMjqrjIAyQ2d5YyCcPwgbllSb8F9PYCCu3CQAA'
    ),
}

for rel_path, encoded_payload in SYNC_PAYLOADS.items():
    target_path = PROJECT_ROOT / rel_path
    target_path.parent.mkdir(parents=True, exist_ok=True)
    target_path.write_bytes(gzip.decompress(base64.b64decode(encoded_payload)))
    print(f'Synchronized {rel_path}')

Synchronized hw14_scripts/kamp_hw14.py
Synchronized hw14_scripts/hw14_experiment_runner.py
Synchronized hw14_scripts/verification_scripts/verify_group_b.py


In [10]:
from contextlib import redirect_stderr, redirect_stdout

from datetime import datetime

import importlib

import shutil

import subprocess

import hw14_experiment_runner
hw14_experiment_runner = importlib.reload(hw14_experiment_runner)

import kamp_hw14
kamp_hw14 = importlib.reload(kamp_hw14)

from kamp_hw14 import run_group_b_structured_tts
from hw14_data_utils import CHECKPOINT_PATH, TeeLogger, load_checkpoint, save_checkpoint, save_text_artifact

GROUP_B_ROOT = PROJECT_ROOT / 'hw14_experiments' / 'group_b_structured_tts'
FULL_METADATA_ROOT = GROUP_B_ROOT / 'full'
FULL_LOG_PATH = PROJECT_ROOT / 'hw14_printouts' / 'group_b_structured_tts_full_log.txt'
FULL_SUMMARY_PATH = GROUP_B_ROOT / 'group_b_full_summary.json'
VERIFY_SCRIPT = PROJECT_ROOT / 'hw14_scripts' / 'verification_scripts' / 'verify_group_b.py'

GROUP_B_ROOT.mkdir(parents=True, exist_ok=True)
FULL_METADATA_ROOT.mkdir(parents=True, exist_ok=True)
FULL_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f'hw14_experiment_runner imported from: {hw14_experiment_runner.__file__}')
print(f'kamp_hw14 imported from: {kamp_hw14.__file__}')
print(f'GROUP_B_ROOT = {GROUP_B_ROOT}')
print(f'FULL_LOG_PATH = {FULL_LOG_PATH}')

hw14_experiment_runner imported from: /content/drive/MyDrive/kamp_hw14/hw14_scripts/hw14_experiment_runner.py
kamp_hw14 imported from: /content/drive/MyDrive/kamp_hw14/hw14_scripts/kamp_hw14.py
GROUP_B_ROOT = /content/drive/MyDrive/kamp_hw14/hw14_experiments/group_b_structured_tts
FULL_LOG_PATH = /content/drive/MyDrive/kamp_hw14/hw14_printouts/group_b_structured_tts_full_log.txt


In [6]:
summary = None

with TeeLogger(FULL_LOG_PATH) as tee, redirect_stdout(tee), redirect_stderr(tee):
    print('[NOTEBOOK] Starting Group B full run')
    summary = run_group_b_structured_tts(mode='full')
    print(summary)

save_text_artifact(FULL_SUMMARY_PATH, summary, as_json=True)
print(summary)

[NOTEBOOK] Starting Group B full run
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/232 [00:00<?, ?B/s]

spm_char.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/413 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

[timer] ga20g_baseline: 0.529s
[audio] Saved ga20g_baseline -> /content/drive/MyDrive/kamp_hw14/hw14_experiments/group_b_structured_tts/exp1_baselines/ga20g_baseline/ga20g_baseline.wav @ 16000 Hz
[timer] ga20f_exp5_llamas_helper_embedding: 0.859s
[audio] Saved ga20f_exp5_llamas_helper_embedding -> /content/drive/MyDrive/kamp_hw14/hw14_experiments/group_b_structured_tts/exp5_speecht5_tts/ga20f_exp5_llamas_helper_embedding.wav @ 16000 Hz
[timer] ga20f_exp5_llamas_zero_vector: 0.644s
[audio] Saved ga20f_exp5_llamas_zero_vector -> /content/drive/MyDrive/kamp_hw14/hw14_experiments/group_b_structured_tts/exp5_speecht5_tts/ga20f_exp5_llamas_zero_vector.wav @ 16000 Hz
[timer] ga20f_exp5_hello_dog_helper_embedding: 0.628s
[audio] Saved ga20f_exp5_hello_dog_helper_embedding -> /content/drive/MyDrive/kamp_hw14/hw14_experiments/group_b_structured_tts/exp5_speecht5_tts/ga20f_exp5_hello_dog_helper_embedding.wav @ 16000 Hz
[timer] ga20f_exp5_hello_dog_zero_vector: 0.549s
[audio] Saved ga20f_exp5_hell

In [7]:
subprocess.run([sys.executable, str(VERIFY_SCRIPT)], check=True)

CompletedProcess(args=['/usr/bin/python3', '/content/drive/MyDrive/kamp_hw14/hw14_scripts/verification_scripts/verify_group_b.py'], returncode=0)

In [8]:
bundle_root = PROJECT_ROOT / 'hw14_experiments' / '_group_b_bundle'
if bundle_root.exists():
    shutil.rmtree(bundle_root)

logs_dir = bundle_root / 'logs'
logs_dir.mkdir(parents=True, exist_ok=True)
group_bundle_root = bundle_root / 'group_b_structured_tts'
shutil.copytree(GROUP_B_ROOT, group_bundle_root, dirs_exist_ok=True)

checkpoint_snapshot = load_checkpoint(CHECKPOINT_PATH)
(group_bundle_root / 'full').mkdir(parents=True, exist_ok=True)
save_checkpoint(checkpoint_snapshot, group_bundle_root / 'full' / 'checkpoint_snapshot.json')

for extra_path in [FULL_LOG_PATH, PROJECT_ROOT / 'hw14_printouts' / 'kamp_hw14_execution_log.txt']:
    if extra_path.exists():
        shutil.copy2(extra_path, logs_dir / extra_path.name)

if FULL_SUMMARY_PATH.exists():
    shutil.copy2(FULL_SUMMARY_PATH, group_bundle_root / FULL_SUMMARY_PATH.name)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
archive_base = PROJECT_ROOT / 'hw14_experiments' / f'group_b_results_{timestamp}'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=bundle_root)
print(f'Created ZIP: {archive_path}')

Created ZIP: /content/drive/MyDrive/kamp_hw14/hw14_experiments/group_b_results_20260404_065638.zip


In [9]:
try:
    from google.colab import files
    files.download(archive_path)
except Exception as error:
    print(f'Download step skipped: {error}')
    print(f'Manually download: {archive_path}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>